In [2]:
from pathlib import Path
import pandas as pd
import numpy as np
import statsmodels.formula.api as smf
from sklearn.metrics import mean_squared_error

DATA_DIR = Path('./type3_sample_csvs')
print('설정된 데이터 폴더:', DATA_DIR)

required_files = ['retail_sales_reg1.csv', 'weather_reg2.csv']

print('[파일 존재 여부 확인]')
for fname in required_files:
    fpath = DATA_DIR / fname
    print(f'{fname}:', 'OK' if fpath.exists() else '없음')

설정된 데이터 폴더: type3_sample_csvs
[파일 존재 여부 확인]
retail_sales_reg1.csv: OK
weather_reg2.csv: OK


---
# 실습 1. `retail_sales_reg1.csv`
## 다중회귀 01: 회귀계수·유의성·R² 해석

## 1-1. 도입 설명

한 소매업체의 월매출(`sales`)에 영향을 미치는 요인을 분석하려고 합니다.
설명변수로는 다음 3개를 사용합니다.

- 광고비: `ad_cost`
- 직원 수: `staff`
- 주말행사 횟수: `event_cnt`

이 실습에서 학생들은 다음을 확인해야 합니다.

1. 회귀식은 어떻게 만들어지는가?
2. 각 회귀계수는 무엇을 의미하는가?
3. p-value를 보고 어떤 변수가 유의한지 어떻게 판단하는가?
4. R²는 무엇을 의미하는가?

---

## 1-2. 분석 전에 기억할 것

- 회귀계수는 **다른 변수가 고정되었을 때**, 해당 변수가 1 증가하면 종속변수가 평균적으로 얼마나 변하는지를 의미합니다.
- p-value < 0.05 이면 보통 **통계적으로 유의한 변수**라고 판단합니다.
- R²는 모델이 종속변수 변동을 얼마나 설명하는지 보여줍니다.

In [3]:
# 1-3. 데이터 불러오기
file_path = DATA_DIR / 'retail_sales_reg1.csv'
df = pd.read_csv(file_path)

print('[데이터 상위 5행]')
print(df.head())
print('[기본 정보]')
print(df.info())
print('[기초통계]')
print(df.describe())

# 1-4. 다중회귀모형 적합
# sales ~ ad_cost + staff + event_cnt
model = smf.ols('sales ~ ad_cost + staff + event_cnt', data=df).fit()

print('[회귀 요약 결과]')
print(model.summary())

# 1-5. 핵심 결과만 따로 보기
print('[회귀계수]')
print(model.params)

print('[p-value]')
print(model.pvalues)

print('[R-squared]')
print(model.rsquared)

print('[95% 신뢰구간]')
print(model.conf_int())

# 1-6. 광고비 10단위 증가 효과 해석 예시
beta_ad = model.params['ad_cost']
change_ad_10 = beta_ad * 10

print('광고비 계수:', beta_ad)
print('광고비 10단위 증가 시 예상 매출 변화:', change_ad_10)

# 유의한 변수 목록
sig_vars = model.pvalues[model.pvalues < 0.05].index.tolist()
print('유의한 변수:', sig_vars)

[데이터 상위 5행]
    sales  ad_cost  staff  event_cnt
0  344.42    38.61     12          4
1  316.86    49.40     14          3
2  290.08    49.31      5          4
3  356.27    46.20     15          4
4  273.46    50.36     13          2
[기본 정보]
<class 'pandas.DataFrame'>
RangeIndex: 80 entries, 0 to 79
Data columns (total 4 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   sales      80 non-null     float64
 1   ad_cost    80 non-null     float64
 2   staff      80 non-null     int64  
 3   event_cnt  80 non-null     int64  
dtypes: float64(2), int64(2)
memory usage: 2.6 KB
None
[기초통계]
            sales    ad_cost      staff  event_cnt
count   80.000000  80.000000  80.000000  80.000000
mean   326.516250  63.250000  12.337500   1.912500
std     76.493853  22.107343   5.279393   1.370519
min    155.950000  26.760000   5.000000   0.000000
25%    274.217500  45.940000   7.000000   1.000000
50%    329.670000  65.495000  13.000000   2.000000
75%    3

---
# 실습 2. `weather_reg2.csv`
## 다중회귀 02: 예측값·신뢰구간·오차지표 해석

## 2-1. 도입 설명

환경 데이터 분석팀은 기온(`temperature`)을 설명하기 위해 아래 설명변수를 사용합니다.

- 일사량: `solar`
- 바람: `wind`
- 오존: `o3`

이 실습에서는 단순히 회귀계수만 보는 것이 아니라,

1. 특정 조건에서의 **예측값**
2. **신뢰구간**
3. **MSE/RMSE**
4. 모델의 설명력(R²)

까지 함께 해석하는 연습을 합니다.

---

## 2-2. 분석 전에 기억할 것

- `predict()`는 예측값을 계산합니다.
- `get_prediction().summary_frame()`은 예측값과 함께 신뢰구간 관련 정보를 제공합니다.
- MSE는 오차의 제곱 평균, RMSE는 그 제곱근입니다.
- RMSE는 원래 종속변수와 같은 단위이므로 해석이 더 직관적입니다.

In [4]:
# 2-3. 데이터 불러오기
file_path = DATA_DIR / 'weather_reg2.csv'
df = pd.read_csv(file_path)

print('[데이터 상위 5행]')
print(df.head())
print('[기본 정보]')
print(df.info())
print('[기초통계]')
print(df.describe())

# 2-4. 다중회귀모형 적합
model = smf.ols('temperature ~ solar + wind + o3', data=df).fit()

print('[회귀 요약 결과]')
print(model.summary())

# 2-5. 핵심 결과만 따로 보기
print('[회귀계수]')
print(model.params)

print('[p-value]')
print(model.pvalues)

print('[R-squared]')
print(model.rsquared)

print('[95% 신뢰구간]')
print(model.conf_int())

# 2-6. 특정 조건에서의 예측값 계산
new_df = pd.DataFrame({
    'solar': [100],
    'wind': [5],
    'o3': [30]
})

pred = model.predict(new_df)
print('[예측값]')
print(pred)

# 2-7. 예측 요약표(평균 예측과 구간 정보)
pred_frame = model.get_prediction(new_df).summary_frame(alpha=0.05)

print('[예측 요약표]')
print(pred_frame)

# 2-8. 오차지표 계산
# 실제값과 모델 예측값 비교

y_true = df['temperature']
y_pred = model.predict(df)

mse = mean_squared_error(y_true, y_pred)
rmse = np.sqrt(mse)

print('[오차지표]')
print('MSE :', mse)
print('RMSE:', rmse)

[데이터 상위 5행]
   temperature   solar  wind     o3
0        14.45   61.16  4.12  16.67
1        17.75  108.21  7.18  30.84
2        20.31  123.01  9.71  45.34
3        20.33  172.55  9.62  53.31
4        18.69  196.61  5.82  25.85
[기본 정보]
<class 'pandas.DataFrame'>
RangeIndex: 150 entries, 0 to 149
Data columns (total 4 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   temperature  150 non-null    float64
 1   solar        150 non-null    float64
 2   wind         150 non-null    float64
 3   o3           150 non-null    float64
dtypes: float64(4)
memory usage: 4.8 KB
None
[기초통계]
       temperature       solar        wind          o3
count   150.000000  150.000000  150.000000  150.000000
mean     21.966333  180.878133    6.131733   32.380133
std       6.368432   72.173137    3.335212   16.213730
min       2.590000   51.130000    1.010000    5.130000
25%      17.370000  124.252500    3.100000   18.415000
50%      22.130000  183.865000    6.0